In [5]:
# Loading in packages and data for MLB team wins prediction model

import pandas as pd

df = pd.read_csv('../data/Teams.csv')
print(df.head())
print(df.columns.tolist())

   yearID lgID teamID franchID divID  Rank   G  Ghome   W   L  ...    DP  \
0    1871  NaN    BS1      BNA   NaN     3  31    NaN  20  10  ...  24.0   
1    1871  NaN    CH1      CNA   NaN     2  28    NaN  19   9  ...  16.0   
2    1871  NaN    CL1      CFC   NaN     8  29    NaN  10  19  ...  15.0   
3    1871  NaN    FW1      KEK   NaN     7  19    NaN   7  12  ...   8.0   
4    1871  NaN    NY2      NNA   NaN     5  33    NaN  16  17  ...  14.0   

      FP                     name                          park  attendance  \
0  0.834     Boston Red Stockings           South End Grounds I         NaN   
1  0.829  Chicago White Stockings       Union Base-Ball Grounds         NaN   
2  0.818   Cleveland Forest Citys  National Association Grounds         NaN   
3  0.803     Fort Wayne Kekiongas                Hamilton Field         NaN   
4  0.840         New York Mutuals      Union Grounds (Brooklyn)         NaN   

     BPF    PPF  teamIDBR  teamIDlahman45  teamIDretro  
0  103.0   

In [7]:
# Keeping needed columns for model and filtering data to 2000-2025 Seasons
cols = [
    'yearID',
    'teamID',
    'W',
    'L',
    'R',
    'RA',
    'HR',
    'SO',
    'ERA',
    'E',
    'attendance'
]

df = df[cols]
df = df[df['yearID'] >= 2000]
print(df.head())
print(df.shape)

      yearID teamID   W   L      R     RA     HR      SO   ERA      E  \
2834    2000    ANA  82  80  864.0  869.0  236.0  1024.0  5.00  134.0   
2835    2000    ARI  85  77  792.0  754.0  179.0   975.0  4.35  107.0   
2836    2000    ATL  95  67  810.0  714.0  179.0  1010.0  4.05  129.0   
2837    2000    BAL  74  88  794.0  913.0  184.0   900.0  5.37  116.0   
2838    2000    BOS  85  77  792.0  745.0  167.0  1019.0  4.23  109.0   

      attendance  
2834   2066982.0  
2835   2942251.0  
2836   3234304.0  
2837   3297031.0  
2838   2585895.0  
(780, 11)


In [9]:
# Check and Handle Missing Values
print(df.isnull().sum())
# Filling missing values with median for numeric columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)
# Dropping any remaining rows with missing values
df = df.dropna()
print(df.isnull().sum())
print(df.shape)

yearID        0
teamID        0
W             0
L             0
R             0
RA            0
HR            0
SO            0
ERA           0
E             0
attendance    0
dtype: int64
yearID        0
teamID        0
W             0
L             0
R             0
RA            0
HR            0
SO            0
ERA           0
E             0
attendance    0
dtype: int64
(780, 11)


C:\Users\tcbailey\AppData\Local\Temp\ipykernel_17748\2676349327.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
C:\Users\tcbailey\AppData\Local\Temp\ipykernel_17748\2676349327.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assign

In [10]:
# Create New Features
df['run_diff'] = df['R'] - df['RA']
df['win_pct'] = df['W'] / (df['W'] + df['L'])
print(df.head())

      yearID teamID   W   L      R     RA     HR      SO   ERA      E  \
2834    2000    ANA  82  80  864.0  869.0  236.0  1024.0  5.00  134.0   
2835    2000    ARI  85  77  792.0  754.0  179.0   975.0  4.35  107.0   
2836    2000    ATL  95  67  810.0  714.0  179.0  1010.0  4.05  129.0   
2837    2000    BAL  74  88  794.0  913.0  184.0   900.0  5.37  116.0   
2838    2000    BOS  85  77  792.0  745.0  167.0  1019.0  4.23  109.0   

      attendance  run_diff   win_pct  
2834   2066982.0      -5.0  0.506173  
2835   2942251.0      38.0  0.524691  
2836   3234304.0      96.0  0.586420  
2837   3297031.0    -119.0  0.456790  
2838   2585895.0      47.0  0.524691  


In [11]:
# Inspecting Final Dataset
print(df.head())
print(df.describe())

      yearID teamID   W   L      R     RA     HR      SO   ERA      E  \
2834    2000    ANA  82  80  864.0  869.0  236.0  1024.0  5.00  134.0   
2835    2000    ARI  85  77  792.0  754.0  179.0   975.0  4.35  107.0   
2836    2000    ATL  95  67  810.0  714.0  179.0  1010.0  4.05  129.0   
2837    2000    BAL  74  88  794.0  913.0  184.0   900.0  5.37  116.0   
2838    2000    BOS  85  77  792.0  745.0  167.0  1019.0  4.23  109.0   

      attendance  run_diff   win_pct  
2834   2066982.0      -5.0  0.506173  
2835   2942251.0      38.0  0.524691  
2836   3234304.0      96.0  0.586420  
2837   3297031.0    -119.0  0.456790  
2838   2585895.0      47.0  0.524691  
            yearID           W           L           R           RA  \
count   780.000000  780.000000  780.000000  780.000000   780.000000   
mean   2012.500000   79.010256   79.010256  719.180769   719.180769   
std       7.504812   15.413260   15.399595  119.510907   124.222478   
min    2000.000000   19.000000   17.000000 

In [12]:
# Save the cleaned dataset for modeling
df.to_csv('../data/mlb_team_wins_cleaned.csv', index=False)